# Job build · étape 2 : train (entraînement simulé)

**Côté :** dev (fabriquer une version).

**Ce que fait cette étape :** aucun calcul GPU ni fine-tuning. Le « modèle » est un prompt (système + template) plus des paramètres de génération, autour d'un endpoint Foundation Model existant. On le fige comme modèle MLflow, on l'enregistre comme **nouvelle version du modèle dev** et on pose l'alias `candidate`. Le job s'arrête là : la promotion appartient à un autre job.

**Ce qu'elle lit :** les paramètres du job (`prompt_variant`, `system_prompt`, `temperature`, `max_tokens`) et la table `faq_knowledge`.

**Ce qu'elle écrit dans MLflow :**
* un run « train » : paramètres, prompt système, template et FAQ en artefacts, modèle logué ;
* une nouvelle version de `<catalog>.<dev_schema>.<model_name>` (jamais d'écrasement) ;
* l'alias `candidate` et les tags `prompt_variant`, `source_run_id`, `validation_status=pending`.

In [0]:
import os, sys
sys.path.append(os.path.abspath(".."))  # rend src/common importable depuis src/build
import mlflow
import pandas as pd
from mlflow import MlflowClient
from mlflow.models.resources import DatabricksServingEndpoint
from common.params import get_param
from common.prompts import build_prompt_config

catalog = get_param(dbutils, "catalog")
dev_schema = get_param(dbutils, "dev_schema")
model_name = get_param(dbutils, "model_name")
serving_endpoint = get_param(dbutils, "serving_endpoint")
experiment_id = get_param(dbutils, "experiment_id")
prompt_variant = get_param(dbutils, "prompt_variant")
system_prompt_override = get_param(dbutils, "system_prompt", required=False)
temperature = float(get_param(dbutils, "temperature"))
max_tokens = int(get_param(dbutils, "max_tokens"))

dev_model = f"{catalog}.{dev_schema}.{model_name}"
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_id=experiment_id)
client = MlflowClient()

In [0]:
prompt = build_prompt_config(prompt_variant, system_prompt_override)
facts = spark.table(f"{catalog}.{dev_schema}.faq_knowledge").orderBy("supplier_id").select("fact").toPandas()["fact"]
faq_text = "\n".join(f"- {f}" for f in facts) if prompt["include_faq"] else ""

model_config = {
    "system_prompt": prompt["system_prompt"],
    "prompt_template": prompt["prompt_template"],
    "faq_text": faq_text,                  # la « connaissance » est figée dans la version
    "serving_endpoint": serving_endpoint,
    "temperature": temperature,
    "max_tokens": max_tokens,
}

In [0]:
with mlflow.start_run(run_name=f"train-{prompt_variant}") as run:
    mlflow.set_tags({"lifecycle_stage": "train", "prompt_variant": prompt_variant})
    mlflow.log_params({
        "prompt_variant": prompt_variant,
        "serving_endpoint": serving_endpoint,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "faq_included": prompt["include_faq"],
    })
    mlflow.log_text(prompt["system_prompt"], "prompt/system_prompt.txt")
    mlflow.log_text(prompt["prompt_template"], "prompt/prompt_template.txt")
    mlflow.log_text(faq_text, "prompt/faq_text.txt")

    model_info = mlflow.pyfunc.log_model(
        name="faq_assistant",
        python_model=os.path.abspath("../model/faq_assistant.py"),  # modèle "from code"
        model_config=model_config,
        input_example=pd.DataFrame({"question": ["Which supplier provides oats?"]}),
        # Déclare l'endpoint utilisé : permet le passage d'identité si le modèle est servi plus tard.
        resources=[DatabricksServingEndpoint(endpoint_name=serving_endpoint)],
        pip_requirements=[f"mlflow-skinny[databricks]=={mlflow.__version__}"],
        registered_model_name=dev_model,   # crée une NOUVELLE version à chaque run
    )

version = str(model_info.registered_model_version)

In [0]:
# CONTRÔLE DE GOUVERNANCE : "candidate" signifie « à évaluer », jamais « utilisable ».
# Le job build n'écrit QUE dans le schéma dev : il n'a besoin d'aucun droit sur prd.
client.set_registered_model_alias(dev_model, "candidate", version)
for key, value in {
    "prompt_variant": prompt_variant,
    "source_run_id": run.info.run_id,
    "validation_status": "pending",
}.items():
    client.set_model_version_tag(dev_model, version, key, value)

dbutils.jobs.taskValues.set(key="model_version", value=version)
print(f"{dev_model} version {version} enregistrée avec l'alias 'candidate'.")
print(f"Pour la promouvoir : lancer le job promote_model avec model_version={version}")